# Feature Ingneering
### Features Implemented:
1. Real time Averge Delay, Same Day, Same Airport (needs reform)
2. sheduled Touraround Time
3. closness to US Holiday.
4. Weather Data (preipation, temperature, windspead)
5. Real time information, 2h before the flight
  - is aircraft here already?
  - was the previous flight of the aircraft delayed?
  - The diffrence between Sceduled Turnaround time, and the delay situation.
### More Feature Ideas:


### Feature ideas Nick:
- Historic rolling avg of delay (7d, 30d)
- Lag-1, Lag-7 (daily avg yesterday, last same weekday avg)
- has_prev_flight
- https://pypi.org/project/holidays/

In [1]:
pip install meteostat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.5/506.5 kB 28.1 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requi

In [ ]:
# mount Google drive
from google.colab import drive
# load the liabries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import meteostat as ms
# This will list the files in your specific folder
drive.mount('/content/drive')
# the list of airports we restirc ourselves to
airport_limit_list =["JFK","LAX","MIA","SFO","EWR","ORD","ATL","DFW","IAH",
"BOS","MCO","FLL","SEA","CLT","DEN","PHL","LAS","HNL","DTW","MSP","PHX","LGA","TPA",
"SLC","BWI","AUS","SAN","HOU","PDX","MDW","OAK","BNA","DCA","STL","DAL"]
# the airlines are already filterd to the ones only that we use.
source = "/content/drive/MyDrive/Datamining/Data_level1/" # set folders for laoding the Data from
destination = "/content/drive/MyDrive/Datamining/Feature_ingeneered_Data/" # where to save the data with new Features
output_file_name = "feature_ingeneared.feather"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
Dataset_name = "1_year_data.parquet"
# load the dataset
dataset = pd.read_parquet(source + Dataset_name)
num_rows = len(dataset)
# iterate through all the columsn and if there are any nan values, we print the column name and the number of nan values

# cut of year, from where we will start using the data for training, before that is just used for feature engneering.
min_year = 2014
dataset["FlightID"] = dataset.index
dataset['DayOfYear'] = dataset['CRSDepDateTime'].dt.dayofyear
# information_time UTC
dataset['Information_time_UTC'] = (dataset['CRSDepDateTime_UTC'] - pd.Timedelta(hours=2))


display(dataset.head())

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,CRSArrDateTime,CRSArrDateTime_UTC,DepDateTime,DepDateTime_UTC,ArrDateTime,ArrDateTime_UTC,knownWeatherDateTime_UTC,FlightID,DayOfYear,Information_time_UTC
0,2014,1,30,4,2014-01-30,AA,N006AA,2377,DFW,ICT,...,2014-01-30 10:55:00,2014-01-30 16:55:00+00:00,2014-01-30 09:35:00,2014-01-30 15:35:00+00:00,2014-01-30 10:51:00,2014-01-30 16:51:00+00:00,2014-01-30 13:00:00,0,30,2014-01-30 13:40:00+00:00
1,2014,1,31,5,2014-01-31,AA,N003AA,2377,DFW,ICT,...,2014-01-31 10:55:00,2014-01-31 16:55:00+00:00,2014-01-31 09:51:00,2014-01-31 15:51:00+00:00,2014-01-31 11:15:00,2014-01-31 17:15:00+00:00,2014-01-31 13:00:00,1,31,2014-01-31 13:40:00+00:00
2,2014,1,1,3,2014-01-01,AA,N002AA,2377,ICT,DFW,...,2014-01-01 13:00:00,2014-01-01 19:00:00+00:00,2014-01-01 11:44:00,2014-01-01 17:44:00+00:00,2014-01-01 13:02:00,2014-01-01 19:02:00+00:00,2014-01-01 15:00:00,2,1,2014-01-01 15:35:00+00:00
3,2014,1,2,4,2014-01-02,AA,N002AA,2377,ICT,DFW,...,2014-01-02 13:00:00,2014-01-02 19:00:00+00:00,2014-01-02 11:34:00,2014-01-02 17:34:00+00:00,2014-01-02 12:53:00,2014-01-02 18:53:00+00:00,2014-01-02 15:00:00,3,2,2014-01-02 15:35:00+00:00
4,2014,1,3,5,2014-01-03,AA,N014AA,2377,ICT,DFW,...,2014-01-03 13:00:00,2014-01-03 19:00:00+00:00,2014-01-03 11:29:00,2014-01-03 17:29:00+00:00,2014-01-03 12:44:00,2014-01-03 18:44:00+00:00,2014-01-03 15:00:00,4,3,2014-01-03 15:35:00+00:00


In [ ]:
# features to calcualte the averge delay fro each departure airport, in every month
df_delays = dataset.groupby(['Origin', dataset['Month'], dataset['Year']])['ArrDelay'].mean().reset_index()
# change the name of the columns
df_delays = df_delays.rename(columns={'ArrDelay': 'prev_AvgArrDelay'})
# fill the nan values with 0, as if there are no flights in the previous month, we can assume that the average delay is 0
# change the key, to the next month, so our feature will be the average delay of the previous month, as we can not use the average delay of the current month, as it would be a data leak
df_delays['Month'] = df_delays['Month'] + 1
mask = (df_delays['Month'] > 12).astype(int)
df_delays['Year'] = df_delays['Year'] + mask
df_delays['Month'] = df_delays['Month'] - mask * 12


# we merge the average delay with the original dataset, on the Origin, month and year of the departure time
dataset = dataset.merge(df_delays, left_on=['Origin', dataset['CRSDepDateTime'].dt.month, dataset['CRSDepDateTime'].dt.year], right_on=['Origin', 'Month', 'Year'], how='left',suffixes=('', '_AvgDelay'))
# we drop the month and year columns
dataset = dataset.drop(columns=['Month_AvgDelay', 'Year_AvgDelay'])
# fillna with the mean
mean = dataset['prev_AvgArrDelay'].mean()
dataset['prev_AvgArrDelay'] = dataset['prev_AvgArrDelay'].fillna(mean)
# display the dataset
display(dataset)

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,CRSArrDateTime_UTC,DepDateTime,DepDateTime_UTC,ArrDateTime,ArrDateTime_UTC,knownWeatherDateTime_UTC,FlightID,DayOfYear,Information_time_UTC,prev_AvgArrDelay
0,2014,1,30,4,2014-01-30,AA,N006AA,2377,DFW,ICT,...,2014-01-30 16:55:00+00:00,2014-01-30 09:35:00,2014-01-30 15:35:00+00:00,2014-01-30 10:51:00,2014-01-30 16:51:00+00:00,2014-01-30 13:00:00,0,30,2014-01-30 13:40:00+00:00,7.010183
1,2014,1,31,5,2014-01-31,AA,N003AA,2377,DFW,ICT,...,2014-01-31 16:55:00+00:00,2014-01-31 09:51:00,2014-01-31 15:51:00+00:00,2014-01-31 11:15:00,2014-01-31 17:15:00+00:00,2014-01-31 13:00:00,1,31,2014-01-31 13:40:00+00:00,7.010183
2,2014,1,1,3,2014-01-01,AA,N002AA,2377,ICT,DFW,...,2014-01-01 19:00:00+00:00,2014-01-01 11:44:00,2014-01-01 17:44:00+00:00,2014-01-01 13:02:00,2014-01-01 19:02:00+00:00,2014-01-01 15:00:00,2,1,2014-01-01 15:35:00+00:00,7.010183
3,2014,1,2,4,2014-01-02,AA,N002AA,2377,ICT,DFW,...,2014-01-02 19:00:00+00:00,2014-01-02 11:34:00,2014-01-02 17:34:00+00:00,2014-01-02 12:53:00,2014-01-02 18:53:00+00:00,2014-01-02 15:00:00,3,2,2014-01-02 15:35:00+00:00,7.010183
4,2014,1,3,5,2014-01-03,AA,N014AA,2377,ICT,DFW,...,2014-01-03 19:00:00+00:00,2014-01-03 11:29:00,2014-01-03 17:29:00+00:00,2014-01-03 12:44:00,2014-01-03 18:44:00+00:00,2014-01-03 15:00:00,4,3,2014-01-03 15:35:00+00:00,7.010183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3417343,2014,12,31,3,2014-12-31,DL,N900DE,2655,JFK,RSW,...,2014-12-31 16:29:00+00:00,2014-12-31 07:50:00,2014-12-31 12:50:00+00:00,2014-12-31 11:05:00,2014-12-31 16:05:00+00:00,2014-12-31 11:00:00,3417343,365,2014-12-31 11:00:00+00:00,-0.021381
3417344,2014,12,31,3,2014-12-31,DL,N386DA,2656,SLC,LAX,...,2014-12-31 16:30:00+00:00,2014-12-31 07:34:00,2014-12-31 14:34:00+00:00,2014-12-31 08:28:00,2014-12-31 16:28:00+00:00,2014-12-31 12:00:00,3417344,365,2014-12-31 12:22:00+00:00,-0.856687
3417345,2014,12,31,3,2014-12-31,DL,N3762Y,2658,LAS,JFK,...,2015-01-01 00:49:00+00:00,2014-12-31 11:51:00,2014-12-31 19:51:00+00:00,2014-12-31 19:15:00,2015-01-01 00:15:00+00:00,2014-12-31 17:00:00,3417345,365,2014-12-31 17:59:00+00:00,4.015764
3417346,2014,12,31,3,2014-12-31,DL,N954DL,2659,FLL,JFK,...,2015-01-01 03:12:00+00:00,2014-12-31 19:14:00,2015-01-01 00:14:00+00:00,2014-12-31 21:57:00,2015-01-01 02:57:00+00:00,2014-12-31 22:00:00,3417346,365,2014-12-31 22:20:00+00:00,0.408429


## Information about real time Averages of delays that day
this cell, adds information about the traffic flow, at the Origin airport, in 2h Before the departure of a flight.
(The last full hour before the 2h before sheduled takeoff Threashold.)
information added:
- number of actually departed flights, that hour
- number of Canncelled Flights that our
- Avergae Dewlay that hour

In [ ]:
# only get this information for the rows we actually need
dataset_airport_limit = dataset[
    dataset['Origin'].isin(airport_limit_list)
    & dataset['Dest'].isin(airport_limit_list)
    & (dataset['Cancelled'] == 0)
    & (dataset['Year'] >= min_year)
].copy()


# code is hard to read because of speed optimization

# Source rows used to compute historical delay at each airport/day, limited in collumns to speed up the process
src = dataset[['Origin', 'FlightDate', 'DepDateTime', 'DepDelayMinutes','Cancelled','Year','FlightID']].copy()
# drop all rows where the flight was in a year to early for our dataset, or where the flight was cancelled, as we can not use these flights to calculate the historical delay at each airport/day
src = src[(src['Year'] >= min_year) & (src['Origin'].isin(airport_limit_list))]


# process on getiing cumlative and avergage informations about the flights.
src['DepDelayMinutes'] = src['DepDelayMinutes'].fillna(0)
# Cumulative delay stats per airport-day (Number of Cannceld Flights, number of departed Flights, Cumlative Delay)
src = src.sort_values(['Origin', 'FlightDate', 'DepDateTime'])
grp = src.groupby(['Origin', 'FlightDate'], sort=False) # group the Values Per Origin Airport and Day, to create cumulative values for each group
src['cum_cannceld_flights'] = grp['Cancelled'].cumsum() # the number of canncelled flights before that flight.
src['cum_delay'] = grp['DepDelayMinutes'].cumsum() # the cumulative delay of all flights before, including this flight (in the Group)

# number of flights so far per group
src['tmp_did_depart'] = -(src['Cancelled'] - 1) # turn the cannceld info into if a flight actually happend
src['cum_count'] = grp['tmp_did_depart'].cumsum() # the number of flights before this flight (including this flight), in the group


# calculate the average delay for each airport/day, by dividing the cumulative delay by the cumulative count of flights
# if there are no flights, we set the average delay to 0, to avoid division by zero
src['avg_delay'] = src['cum_delay'] / src['cum_count']
src['avg_delay'] = src['avg_delay'].fillna(0)

# drop cancelled flights, as we can not use these flights to calculate the historical delay at each airport/day
src = src[src['Cancelled'] == 0]

# group the information by Origin, Flight date, and the next hour (ceil hour) of the departure time, to get the cumulative delay and count of flights for each airport, here always use the row where the time is the latest or the count is highest.
src['DepDateTime_ceil'] = src['DepDateTime'].dt.ceil('h') # get the next hour for every flight.

src = src.sort_values(['Origin', 'FlightDate', 'DepDateTime_ceil', 'DepDateTime'], ascending=[True, True, True, False]) # for order to have for every same Ceil hour the flight with the highest minutes number (last flight in the hour) first
src = src.drop_duplicates(subset=['Origin', 'FlightDate', 'DepDateTime_ceil'], keep='first') # only keep the last flight in the hour
grp = src.groupby(['Origin', 'FlightDate', 'DepDateTime_ceil'], sort=False)
# for every hour i want the last information about the cumulative delay and count of flights, so i want to get the last row for each group
src = grp.last().reset_index()
# also get the number of flights that departed in the last hour, to get the cumulative count of flights for each airport, here always use the row where the time is the latest or the count is highest.
src['flights_in_last_hour'] = grp['tmp_did_depart'].sum().reset_index(drop=True) # the last flight in the hour has the information about how manny flights happend in that hour.

# Now the src dataset, now has statsitical information Per Airport and Hour,
# - the number of flights that actually departed in that hour (without canncelled flights)
# - the number of canncelled flights
# - the average Departue Delay of the flights, in that hour
# - the number of

# we will merge these statistical informations onto the dataset.
# we will merge the hour to the hour to the last hour 2h before sheduled Takeoff

# get the last hourmark for every flight before the 2h before takeoff threashold.
hours_before =2
dataset_airport_limit['floor_informationtime'] = dataset_airport_limit['Information_time_UTC'].dt.tz_localize(None) - pd.to_timedelta(hours_before, unit='h')
# the columsn we still need of src:
src = src[['Origin', 'FlightDate', 'DepDateTime_ceil', 'avg_delay', 'cum_count','cum_cannceld_flights', 'flights_in_last_hour']].copy()
# we merge the information of src with the original dataset, by merging on the Origin, FlightDate and the next hour (ceil hour) of the departure time, to get the cumulative delay and count of flights for each airport, here always use the row where the time is the latest or the count is highest.
dataset_airport_limit = dataset_airport_limit.merge(src, left_on=['Origin', 'FlightDate', 'floor_informationtime'], right_on=['Origin', 'FlightDate', 'DepDateTime_ceil'], how='left',suffixes=('', '_src'))
# fill the nan values with 0, as if there are no flights in the previous hour, we can assume that the average delay is 0 and the count of flights is 0
dataset_airport_limit['2h_prev_avg_delay'] = dataset_airport_limit['avg_delay'].fillna(0)
dataset_airport_limit['2h_prev_cum_count'] = dataset_airport_limit['cum_count'].fillna(0)
dataset_airport_limit['2h_prev_cum_cancelled'] = dataset_airport_limit['cum_cannceld_flights'].fillna(0)
dataset_airport_limit['2h_prev_flights_in_last_hour'] = dataset_airport_limit['flights_in_last_hour'].fillna(0)
# display the df

# we drop the columns we do not need anymore
dataset_airport_limit = dataset_airport_limit.drop(columns=['DepDateTime_ceil','floor_informationtime'])
mergeing_collumns = ["FlightID", "2h_prev_avg_delay", "2h_prev_cum_count","2h_prev_cum_cancelled", "2h_prev_flights_in_last_hour"]
dataset_airport_limit = dataset_airport_limit[mergeing_collumns]
display(dataset_airport_limit)
print(dataset_airport_limit[mergeing_collumns].isna().sum())
# we merge the information of src with the original dataset, by merging on the Origin, FlightDate and the next hour (ceil hour) of the departure time, to get the cumulative delay and count of flights for each airport, here always use the row where the time is the latest or the count is highest.
dataset = dataset.merge(dataset_airport_limit, left_on='FlightID', right_on='FlightID', how='left')

# garabe collect the variablex we dont need anymore
del src, grp,  dataset_airport_limit


,FlightID,2h_prev_avg_delay,2h_prev_cum_count,2h_prev_cum_cancelled,2h_prev_flights_in_last_hour
0,33,0.0,0.0,0.0,0.0
1,34,0.0,0.0,0.0,0.0
2,35,0.0,0.0,0.0,0.0
3,36,0.0,0.0,0.0,0.0
4,37,0.0,0.0,0.0,0.0
...,...,...,...,...,...
1872201,3417342,0.0,0.0,0.0,0.0
1872202,3417344,0.0,0.0,0.0,0.0
1872203,3417345,0.0,0.0,0.0,0.0
1872204,3417346,0.0,0.0,0.0,0.0


FlightID                        0
2h_prev_avg_delay               0
2h_prev_cum_count               0
2h_prev_cum_cancelled           0
2h_prev_flights_in_last_hour    0
dtype: int64


In [ ]:
display(dataset[mergeing_collumns])
# display the amount of nans for these collumns
print(dataset[mergeing_collumns].isna().sum())

,FlightID,2h_prev_avg_delay,2h_prev_cum_count,2h_prev_cum_cancelled,2h_prev_flights_in_last_hour
0,0,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN
2,2,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN
4,4,NaN,NaN,NaN,NaN
...,...,...,...,...,...
3417343,3417343,NaN,NaN,NaN,NaN
3417344,3417344,0.0,0.0,0.0,0.0
3417345,3417345,0.0,0.0,0.0,0.0
3417346,3417346,0.0,0.0,0.0,0.0


FlightID                              0
2h_prev_avg_delay               1545142
2h_prev_cum_count               1545142
2h_prev_cum_cancelled           1545142
2h_prev_flights_in_last_hour    1545142
dtype: int64


In [ ]:
# we now start computing the Turnaround time
# we sort the dataset by tail number and flight date, to make it easier to find the previous flight
dataset = dataset.sort_values(by=['Tail_Number', 'CRSDepDateTime_UTC'])
display(dataset)

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,ArrDateTime_UTC,knownWeatherDateTime_UTC,FlightID,DayOfYear,Information_time_UTC,prev_AvgArrDelay,2h_prev_avg_delay,2h_prev_cum_count,2h_prev_cum_cancelled,2h_prev_flights_in_last_hour
222448,2014,1,1,3,2014-01-01,DL,D942DN,2279,MSP,SJC,...,2014-01-02 03:24:00+00:00,2014-01-01 21:00:00,222448,1,2014-01-01 21:35:00+00:00,7.010183,NaN,NaN,NaN,NaN
223073,2014,1,2,4,2014-01-02,DL,D942DN,1056,SJC,MSP,...,2014-01-02 18:19:00+00:00,2014-01-02 12:00:00,223073,2,2014-01-02 12:35:00+00:00,7.010183,NaN,NaN,NaN,NaN
222885,2014,1,2,4,2014-01-02,DL,D942DN,785,MSP,ATL,...,2014-01-02 21:47:00+00:00,2014-01-02 17:00:00,222885,2,2014-01-02 17:15:00+00:00,7.010183,0.0,0.0,0.0,0.0
221429,2014,1,2,4,2014-01-02,DL,D942DN,1307,ATL,RIC,...,2014-01-02 23:58:00+00:00,2014-01-02 18:00:00,221429,2,2014-01-02 18:52:00+00:00,7.010183,NaN,NaN,NaN,NaN
221430,2014,1,2,4,2014-01-02,DL,D942DN,1307,RIC,ATL,...,2014-01-03 02:23:00+00:00,2014-01-02 21:00:00,221430,2,2014-01-02 21:04:00+00:00,7.010183,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3056018,2014,11,1,6,2014-11-01,F9,NaN,949,TTN,MSP,...,NaT,NaT,3056018,305,NaT,-3.412621,NaN,NaN,NaN,NaN
3124879,2014,11,19,3,2014-11-19,F9,NaN,1221,ILG,MCO,...,NaT,NaT,3124879,323,NaT,-4.344262,NaN,NaN,NaN,NaN
3355458,2014,12,21,7,2014-12-21,UA,NaN,200,GUM,HNL,...,NaT,NaT,3355458,355,NaT,18.766667,NaN,NaN,NaN,NaN
3415913,2014,12,30,2,2014-12-30,F9,NaN,901,TTN,ATL,...,NaT,NaT,3415913,364,NaT,19.198953,NaN,NaN,NaN,NaN


In [ ]:
# report if there are any nans int the df
print("Number of NaN values in each column:")
print(dataset.isna().sum())

Number of NaN values in each column:
Year                                     0
Month                                    0
DayofMonth                               0
DayOfWeek                                0
FlightDate                               0
Reporting_Airline                        0
Tail_Number                           9260
Flight_Number_Reporting_Airline          0
Origin                                   0
Dest                                     0
CRSDepTime                               0
DepTime                              42390
DepDelay                             42390
DepDelayMinutes                      42390
TaxiOut                              43280
WheelsOff                            43280
WheelsOn                             51690
TaxiIn                               44584
CRSArrTime                               0
ArrTime                              51690
ArrDelay                             51690
ArrDelayMinutes                      51690
Cancelled        

In [ ]:
# finding the previous flight for each aircraft.
# naivly for now take the FlightID of the row above.
dataset['tmp_PreviousFlightId'] = dataset['FlightID'].shift(1)
# get the arrival airport of the previous flight, from the row above
dataset['PreviousFlightDest'] = dataset['Dest'].shift(1)
#  also get airline and Tail number of the previous flight
dataset['PreviousFlightAirline'] = dataset['Reporting_Airline'].shift(1)
dataset['PreviousFlightTailNumber'] = dataset['Tail_Number'].shift(1)
# also get if the previous flight was cancelled or diverted, to exclude these flights later
dataset['PreviousFlightCancelled'] = dataset['Cancelled'].shift(1)
dataset['PreviousFlightDiverted'] = dataset['Diverted'].shift(1)

# build a mask to see if the airlines match, if the tail number matches and if the departure airport of the current flight matches the arrival airport of the previous flight
mask = (dataset['Tail_Number'] == dataset['PreviousFlightTailNumber']) & (dataset['Reporting_Airline'] == dataset['PreviousFlightAirline']) & (dataset['Origin'] == dataset['PreviousFlightDest'])
# also exclude the previous flight if it was cancelled or diverted
mask = mask & (dataset['PreviousFlightCancelled'] == 0) & (dataset['PreviousFlightDiverted'] == 0)

# apply the mask
dataset['PreviousFlightId'] = dataset['tmp_PreviousFlightId'].where(mask, other=np.nan)

# drop the temporary columns
dataset = dataset.drop(columns=['tmp_PreviousFlightId', 'PreviousFlightDest', 'PreviousFlightAirline', 'PreviousFlightTailNumber', 'PreviousFlightCancelled', 'PreviousFlightDiverted'])


# now that i have the previous flights information we drop the first month of our dataset, as we only wanted it for the previous flight information.
dataset = dataset[dataset['Year'] >= min_year]
# introduce if this is the first flight record for the aircraft, by checking if the PreviousFlightId is null
dataset['FirstFlightRecord'] = dataset['PreviousFlightId'].isna().astype(int)


In [ ]:
import numpy as np
# create a subeset of columns relevant for joingg with the original dataset to get the departure and arrival times of the previous flight
previous_flights = dataset[['FlightID','CRSArrDateTime_UTC','ArrDateTime_UTC','DepDateTime_UTC','CRSDepDateTime_UTC','DepDelayMinutes']]
print("1")
# rename the columns to indicate that they are the departure and arrival times of the previous flight
previous_flights = previous_flights.rename(columns={'CRSArrDateTime_UTC': 'CRSPreviousFlightArrDateTime_UTC', 'ArrDateTime_UTC': 'PreviousFlightArrDateTime_UTC'})
print('2')
# join the previous_flights dataset with the original dataset, to get the departure and arrival times of the previous flight
dataset = dataset.merge(previous_flights, left_on='PreviousFlightId', right_on='FlightID', how='left', suffixes=('', '_PreviousFlight'))
print("3")
# calculate the turnaround time in minutes, by taking the difference between the departure time of the current flight and the arrival time of the previous flight
dataset['CRSTurnaroundTime'] = (dataset['CRSDepDateTime_UTC'] - dataset['CRSPreviousFlightArrDateTime_UTC']).dt.total_seconds() / 60

mean_turnaround_time = dataset['CRSTurnaroundTime'].mean()
dataset['CRSTurnaroundTime'] = dataset['CRSTurnaroundTime'].fillna(mean_turnaround_time)

# === has_prev_flight flag ===
dataset['has_prev_flight'] = dataset['CRSTurnaroundTime'].notna().astype(int)

# opportunity to maybe add furhter feature, that are regarding the last flight of an aircraft


# === IF flight is already confirmed at airport 2h Before the sceduled flight ===

# Localize knownWeatherDateTime_UTC to UTC to make it comparable
dataset["Airplane_already_at_airport"] = (dataset["Information_time_UTC"] >= dataset["PreviousFlightArrDateTime_UTC"]).astype(np.float32)

# if ther is no previous flight, we dont know it so we dont know if the airplane is at the airport already
dataset.loc[(dataset['has_prev_flight']==0),["Airplane_already_at_airport"]] = 0.5

# === information if the previous flight has a delay already, that we can know of, bacuse the departure was more than 2h ago
# the min departure delay of the Previous flight.
# there are 3 cases for the calululation of this feature:
# 1. the Plane was not supposed to Depart already, 2h before the subject flight (informationtime) -> the previous delay is 0
# 2. the plane was supposed to depart already and has -> Departure DelayMinutes
# 3. was supposed to but hasnt yet, than the diffrence from now to the planned DepTime

print("start")
should_have_Departed = (dataset["Information_time_UTC"]>= dataset["CRSDepDateTime_UTC"])
has_departed_mask = (dataset["Information_time_UTC"] >= dataset["DepDateTime_UTC_PreviousFlight"])
# 1.  where it shoudl have departed
dataset["Prev_flight_DelayMinutes"] = np.where(~should_have_Departed, 0, 0)
# 2. where has departed we take the Delay
dataset["Prev_flight_DelayMinutes"] = np.where(has_departed_mask, dataset["DepDelayMinutes_PreviousFlight"], dataset["Prev_flight_DelayMinutes"])
# 3. where should but has not
dataset["Prev_flight_DelayMinutes"] = np.where((should_have_Departed & (~has_departed_mask)), (dataset["Information_time_UTC"] - dataset["CRSDepDateTime_UTC"]).dt.total_seconds() / 60, dataset["Prev_flight_DelayMinutes"])
print("finished")
dataset["Expected_Tournaround_time"] = dataset["CRSTurnaroundTime"] - dataset["Prev_flight_DelayMinutes"]
dataset["Expected_Tournaround_time"] = dataset["Expected_Tournaround_time"].fillna(dataset["CRSTurnaroundTime"])

# === LEAKAGE CHECK ===
# Only keep prev flight info if the previous flight ACTUALLY arrived before our prediction time
# prediction_time = dataset['CRSDepDateTime_UTC'] - pd.Timedelta(hours=2)
# leakage_mask = dataset['PreviousFlightArrDateTime_UTC'].notna() & (dataset['PreviousFlightArrDateTime_UTC'] >= prediction_time)
# print(f"Leakage check: {leakage_mask.sum():,} rows had prev flight not yet arrived — nullified")
# dataset.loc[leakage_mask, ['CRSTurnaroundTime', 'PreviousFlightArrDateTime_UTC', 'PreviousFlightArrDateTime']] = np.nan


print(f"has_prev_flight: {dataset['has_prev_flight'].mean()*100:.1f}% of flights have previous flight data")


display(dataset[['PreviousFlightId', 'Expected_Tournaround_time', 'CRSDepDateTime_UTC', 'PreviousFlightArrDateTime_UTC', 'CRSTurnaroundTime', 'has_prev_flight',"Prev_flight_DelayMinutes","Airplane_already_at_airport","DepDelayMinutes","ArrDelayMinutes"]].head(20))

del previous_flights

1
2
3
start
finished
has_prev_flight: 100.0% of flights have previous flight data


,PreviousFlightId,Expected_Tournaround_time,CRSDepDateTime_UTC,PreviousFlightArrDateTime_UTC,CRSTurnaroundTime,has_prev_flight,Prev_flight_DelayMinutes,Airplane_already_at_airport,DepDelayMinutes,ArrDelayMinutes
0,NaN,255.957037,2014-01-01 23:35:00+00:00,NaT,255.957037,1,0.0,0.0,0.0,0.0
1,222448.0,653.000000,2014-01-02 14:35:00+00:00,2014-01-02 03:24:00+00:00,653.000000,1,0.0,1.0,0.0,0.0
2,223073.0,48.000000,2014-01-02 19:15:00+00:00,2014-01-02 18:19:00+00:00,48.000000,1,0.0,0.0,0.0,4.0
3,222885.0,-51.000000,2014-01-02 20:52:00+00:00,2014-01-02 21:47:00+00:00,-51.000000,1,0.0,0.0,104.0,95.0
4,221429.0,41.000000,2014-01-02 23:04:00+00:00,2014-01-02 23:58:00+00:00,41.000000,1,0.0,0.0,93.0,90.0
5,221430.0,18.000000,2014-01-03 02:44:00+00:00,2014-01-03 02:23:00+00:00,111.000000,1,93.0,0.0,49.0,37.0
6,225538.0,332.000000,2014-01-03 10:40:00+00:00,2014-01-03 04:56:00+00:00,381.000000,1,49.0,1.0,0.0,0.0
7,223379.0,37.000000,2014-01-03 13:10:00+00:00,2014-01-03 12:15:00+00:00,37.000000,1,0.0,0.0,0.0,0.0
8,223382.0,41.000000,2014-01-03 16:08:00+00:00,2014-01-03 15:23:00+00:00,41.000000,1,0.0,0.0,0.0,4.0
9,223383.0,46.000000,2014-01-03 18:49:00+00:00,2014-01-03 18:07:00+00:00,46.000000,1,0.0,0.0,59.0,34.0


In [ ]:
# how manny nan values are in the CRSTurnaroundTime column
num_nan = dataset['CRSTurnaroundTime'].isna().sum()
print("Number of NaN values in the CRSTurnaroundTime column: ", num_nan)

# how manny times the tailnumber is missing in the dataset
num_nan_tailnumber = dataset['Tail_Number'].isna().sum()
print("Number of NaN values in the Tail_Number column: ", num_nan_tailnumber)
# mean tournaround time
mean_turnaround_time = dataset['CRSTurnaroundTime'].mean()
print("Mean turnaround time: ", mean_turnaround_time)

# fill the nan values in the CRSTurnaroundTime column with the mean turnaround time
#dataset['CRSTurnaroundTime'] = dataset['CRSTurnaroundTime'].fillna(mean_turnaround_time)

# Fill NaN with 0 instead of mean — the has_prev_flight flag disambiguates
dataset['has_prev_flight'] = dataset['has_prev_flight'].fillna(0)
# display(dataset)
# show all rows where the turnaround time is less than 0 and the previous row
# set all values for negative DepDelay to 0

planned_neg_tournaround = dataset['CRSTurnaroundTime']  < -15
# drop all instances where the actual turnaround are more than 0
# actual_neg_tournaround = dataset['actual_CRSTurnaroundTime'] < -10



# merge the two mask with an and
neg_tournaround = planned_neg_tournaround

# make a mask for all cancellled flights
cancellled_flights = dataset['Cancelled'] == 1
diverted_flights = dataset['Diverted'] == 1

# remove all cancelled or diverted flights from the neg_tournaround mask
neg_tournaround = neg_tournaround & ~cancellled_flights & ~diverted_flights

# print the number of flights with an actual turnaround time of less than -10 minutes and a planned turnaround time of less than -15 minutes, that are not cancelled or diverted
actual_neg_tournaround2 =  ~cancellled_flights & ~diverted_flights
print("num_actual_neg_tournaround: ", actual_neg_tournaround2.sum())

# apply the combined mask to the dataset
neg_tournaround_dataset = dataset[neg_tournaround]
display(neg_tournaround_dataset)
del neg_tournaround_dataset
# filter the dataset for values of turnaround time that are less than 0, as these are likely to be errors in the data
# neg_tournaround_dataset = dataset[dataset['CRSTurnaroundTime'] <= 0]
# display(neg_tournaround_dataset)
# drop the actual_TurnaroundTime column, as it is not needed anymore
# dataset = dataset.drop(columns=['actual_TurnaroundTime'])


Number of NaN values in the CRSTurnaroundTime column:  0
Number of NaN values in the Tail_Number column:  9260
Mean turnaround time:  255.95703724278818
num_actual_neg_tournaround:  3365658


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,CRSPreviousFlightArrDateTime_UTC,PreviousFlightArrDateTime_UTC,DepDateTime_UTC_PreviousFlight,CRSDepDateTime_UTC_PreviousFlight,DepDelayMinutes_PreviousFlight,CRSTurnaroundTime,has_prev_flight,Airplane_already_at_airport,Prev_flight_DelayMinutes,Expected_Tournaround_time
3,2014,1,2,4,2014-01-02,DL,D942DN,1307,ATL,RIC,...,2014-01-02 21:43:00+00:00,2014-01-02 21:47:00+00:00,2014-01-02 19:11:00+00:00,2014-01-02 19:15:00+00:00,0.0,-51.0,1,0.0,0.0,-51.0
63,2014,1,15,3,2014-01-15,DL,D942DN,729,MSP,ORD,...,2014-01-16 02:57:00+00:00,2014-01-16 02:54:00+00:00,2014-01-16 00:06:00+00:00,2014-01-16 00:05:00+00:00,1.0,-82.0,1,0.0,0.0,-82.0
383,2014,4,11,5,2014-04-11,DL,D942DN,104,ATL,BOS,...,2014-04-11 14:02:00+00:00,2014-04-11 13:53:00+00:00,2014-04-11 11:24:00+00:00,2014-04-11 11:30:00+00:00,0.0,-67.0,1,0.0,0.0,-67.0
404,2014,4,15,2,2014-04-15,DL,D942DN,104,ATL,BOS,...,2014-04-15 14:02:00+00:00,2014-04-15 14:04:00+00:00,2014-04-15 11:27:00+00:00,2014-04-15 11:30:00+00:00,0.0,-67.0,1,0.0,0.0,-67.0
750,2014,6,23,1,2014-06-23,DL,D942DN,528,MSP,MSN,...,2014-06-23 21:42:00+00:00,2014-06-23 21:36:00+00:00,2014-06-23 19:27:00+00:00,2014-06-23 19:30:00+00:00,0.0,-64.0,1,0.0,0.0,-64.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3407339,2014,7,2,3,2014-07-02,DL,N999DN,1485,LGA,MCO,...,2014-07-03 01:16:00+00:00,2014-07-03 03:03:00+00:00,2014-07-03 00:19:00+00:00,2014-07-02 22:45:00+00:00,94.0,-76.0,1,0.0,0.0,-76.0
3407399,2014,7,18,5,2014-07-18,DL,N999DN,2096,MSP,LGA,...,2014-07-18 18:20:00+00:00,2014-07-18 18:01:00+00:00,2014-07-18 15:41:00+00:00,2014-07-18 15:45:00+00:00,0.0,-18.0,1,0.0,0.0,-18.0
3407530,2014,8,19,2,2014-08-19,DL,N999DN,1549,ATL,TYS,...,2014-08-19 20:47:00+00:00,2014-08-19 21:06:00+00:00,2014-08-19 19:21:00+00:00,2014-08-19 19:10:00+00:00,11.0,-19.0,1,0.0,0.0,-19.0
3407890,2014,11,13,4,2014-11-13,DL,N999DN,55,ATL,IAH,...,2014-11-13 14:31:00+00:00,2014-11-13 14:38:00+00:00,2014-11-13 12:22:00+00:00,2014-11-13 12:25:00+00:00,0.0,-16.0,1,0.0,0.0,-16.0


In [ ]:


correlation = dataset['CRSTurnaroundTime'].corr(dataset['DepDelay'])
print("Correlation between turnaround time and departure delay: ", correlation)

# smae thing for the arrival delay
correlation = dataset['CRSTurnaroundTime'].corr(dataset['ArrDelay'])
print("Correlation between turnaround time and arrival delay: ", correlation)

# smae thing for the arrival delay
correlation = dataset['Expected_Tournaround_time'].corr(dataset['ArrDelay'])
print("Correlation between expected_torunaroudn_time and arrival delay: ", correlation)

correlation = dataset['Prev_flight_DelayMinutes'].corr(dataset['ArrDelay'])
print("Correlation between prev_flight Delay and arrival delay: ", correlation)


Correlation between turnaround time and departure delay:  -0.012934068268131729
Correlation between turnaround time and arrival delay:  -0.01146841069967111
Correlation between expected_torunaroudn_time and arrival delay:  -0.012940327913600908
Correlation between prev_flight Delay and arrival delay:  0.06903674486357243


In [ ]:
# list all columns in the dataset
print(dataset.columns)
#  remove the colums that are a refrence to the previous flight, as they are not needed anymore
dataset = dataset.drop(columns=['PreviousFlightId', 'PreviousFlightArrDateTime_UTC', 'CRSPreviousFlightArrDateTime_UTC','FlightID_PreviousFlight','DepDateTime_UTC_PreviousFlight','CRSDepDateTime_UTC_PreviousFlight','DepDelayMinutes_PreviousFlight'])

display(dataset)


Index(['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline',
       'Origin', 'Dest', 'CRSDepTime', 'DepTime', 'DepDelay',
       'DepDelayMinutes', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn',
       'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'Cancelled',
       'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime',
       'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay',
       'SecurityDelay', 'LateAircraftDelay', 'TZ_Origin', 'TZ_Dest',
       'CRSDepDateTime', 'CRSDepDateTime_UTC', 'CRSArrDateTime',
       'CRSArrDateTime_UTC', 'DepDateTime', 'DepDateTime_UTC', 'ArrDateTime',
       'ArrDateTime_UTC', 'knownWeatherDateTime_UTC', 'FlightID', 'DayOfYear',
       'Information_time_UTC', 'prev_AvgArrDelay', '2h_prev_avg_delay',
       '2h_prev_cum_count', '2h_prev_cum_cancelled',
       '2h_prev_flights_in_last_hour', 'PreviousFlightId', 'FirstFlightRecord',
  

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,2h_prev_avg_delay,2h_prev_cum_count,2h_prev_cum_cancelled,2h_prev_flights_in_last_hour,FirstFlightRecord,CRSTurnaroundTime,has_prev_flight,Airplane_already_at_airport,Prev_flight_DelayMinutes,Expected_Tournaround_time
0,2014,1,1,3,2014-01-01,DL,D942DN,2279,MSP,SJC,...,NaN,NaN,NaN,NaN,1,255.957037,1,0.0,0.0,255.957037
1,2014,1,2,4,2014-01-02,DL,D942DN,1056,SJC,MSP,...,NaN,NaN,NaN,NaN,0,653.000000,1,1.0,0.0,653.000000
2,2014,1,2,4,2014-01-02,DL,D942DN,785,MSP,ATL,...,0.0,0.0,0.0,0.0,0,48.000000,1,0.0,0.0,48.000000
3,2014,1,2,4,2014-01-02,DL,D942DN,1307,ATL,RIC,...,NaN,NaN,NaN,NaN,0,-51.000000,1,0.0,0.0,-51.000000
4,2014,1,2,4,2014-01-02,DL,D942DN,1307,RIC,ATL,...,NaN,NaN,NaN,NaN,0,41.000000,1,0.0,0.0,41.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3417343,2014,11,1,6,2014-11-01,F9,NaN,949,TTN,MSP,...,NaN,NaN,NaN,NaN,1,255.957037,1,0.0,0.0,255.957037
3417344,2014,11,19,3,2014-11-19,F9,NaN,1221,ILG,MCO,...,NaN,NaN,NaN,NaN,1,255.957037,1,0.0,0.0,255.957037
3417345,2014,12,21,7,2014-12-21,UA,NaN,200,GUM,HNL,...,NaN,NaN,NaN,NaN,1,255.957037,1,0.0,0.0,255.957037
3417346,2014,12,30,2,2014-12-30,F9,NaN,901,TTN,ATL,...,NaN,NaN,NaN,NaN,1,255.957037,1,0.0,0.0,255.957037


In [ ]:
# reduce the Dataset, to the things we actually want to train with.
dataset = dataset[~dataset['CRSElapsedTime'].isna()]
# we finally remove all the flights that do not land and depart from an airport in the List.
dataset = dataset[dataset["Origin"].isin(airport_limit_list) & dataset["Dest"].isin(airport_limit_list)]
dataset = dataset[~((dataset['Diverted'] == 1) | (dataset['Cancelled'] == 1))]

#### Bejond this point only the Entries intended for training are in the Dataset

> this reduces RAM load, but means that the Possibility for new fetaures is limited, beyond this point.

In [ ]:



columns_with_nan_allowed = ['Tail_Number', 'CRSTurnaroundTime','NASDelay','SecurityDelay','LateAircraftDelay','CarrierDelay','WeatherDelay',]
print(f"length of df: {len(dataset)}")
for col in dataset.columns:
    unique_values = dataset[col].nunique()
    missing_values = dataset[col].isna().sum()
    if missing_values > 0:
        print(f"{col}:  Unique:   {unique_values} ---  Missing: {missing_values}")
        if missing_values > 0 and col not in columns_with_nan_allowed:
            print(f"Column {col} has {missing_values} missing values that are not due to cancelled or diverted flights, which is a problem for the analysis.")


length of df: 1867371
CarrierDelay:  Unique:   804 ---  Missing: 1446660
WeatherDelay:  Unique:   475 ---  Missing: 1446660
NASDelay:  Unique:   454 ---  Missing: 1446660
SecurityDelay:  Unique:   98 ---  Missing: 1446660
LateAircraftDelay:  Unique:   575 ---  Missing: 1446660


### Feature, telling how close we are to the next US Holidy

In [ ]:
# # handwritten list of some US holidays, to introduce a new feature
# # New Year's Day, Martin Luther King Jr. Day, Presidents' Day, Memorial Day, Independence Day, Labor Day, Columbus Day, Veterans Day, Thanksgiving, Christmas eve Christmas day, new years eve
# us_holidays = [
#     "2023-01-01",  # New Year's Day
#     "2023-01-16",  # Martin Luther King Jr. Day
#     "2023-02-20",  # Presidents' Day
#     "2023-05-29",  # Memorial Day
#     "2023-07-04",  # Independence Day
#     "2023-09-04",  # Labor Day
#     "2023-10-09",  # Columbus Day
#     "2023-11-11",  # Veterans Day
#     "2023-11-23",  # Thanksgiving
#     "2023-12-24",  # Christmas Eve
#     "2023-12-25",  # Christmas Day
#     "2023-12-31"   # New Year's Eve
# ]
# # convert the list to datetime
# us_holidays = pd.to_datetime(us_holidays)

# # to speed up the Holiday flag and info for next holiday, we group an the FlightDate
# grp = dataset.groupby('FlightDate')

# # introduce a new feature if the flight is on a holiday or not
# dataset['IsHoliday'] = dataset['CRSDepDateTime'].dt.normalize().isin(us_holidays).astype(int)

# # create a feature for the distance to the nearest holiday, by calculating the difference in days between the flight date and the nearest holiday
# dataset['DaysToNearestHoliday'] = dataset['CRSDepDateTime'].dt.normalize().apply(lambda x: min(abs((x - holiday).days) for holiday in us_holidays))


# ---------------------
# Using holiday library

import holidays

# Generate holiday dates for all years in the dataset
years_in_data = range(int(dataset['Year'].min()), int(dataset['Year'].max()) + 1)
us_holiday_dates = holidays.US(years=years_in_data)
# Add Christmas Eve and New Year's Eve (not in the official US holidays list)
for y in years_in_data:
    us_holiday_dates[pd.Timestamp(y, 12, 24)] = "Christmas Eve"
    us_holiday_dates[pd.Timestamp(y, 12, 31)] = "New Year's Eve"



# holiday_df = pd.DataFrame({'holiday_date': sorted(us_holiday_dates.keys())})
# holiday_df['holiday_date'] = pd.to_datetime(holiday_df['holiday_date']).dt.normalize()

holiday_dates_set = set(pd.to_datetime(list(us_holiday_dates.keys())).normalize())
print(f"Total holiday dates generated: {len(holiday_dates_set)} across {len(list(years_in_data))} years")

# dataset = dataset.sort_values('FlightDate')
dataset['FlightDate'] = pd.to_datetime(dataset['FlightDate']).dt.normalize()

# IsHoliday: binary flag
dataset['IsHoliday'] = dataset['FlightDate'].dt.normalize().isin(holiday_dates_set).astype(int)
print(f"Flights on holidays: {dataset['IsHoliday'].sum():,} ({dataset['IsHoliday'].mean()*100:.1f}%)")

# DaysToNearestHoliday: distance in days to the closest holiday
holidays_arr = np.array(sorted(holiday_dates_set), dtype='datetime64[D]')
flight_dates_arr = dataset['FlightDate'].dt.normalize().values.astype('datetime64[D]')
# find fitting holidays to flightdates
idx = np.searchsorted(holidays_arr, flight_dates_arr)
# handle last and first holdays
idx_prev = np.clip(idx - 1, 0, len(holidays_arr) - 1)
idx_next = np.clip(idx, 0, len(holidays_arr) - 1)

# calculate distances
dist_prev = np.abs((flight_dates_arr - holidays_arr[idx_prev]).astype('timedelta64[D]').astype(int))
dist_next = np.abs((holidays_arr[idx_next] - flight_dates_arr).astype('timedelta64[D]').astype(int))


# extract teh min
dataset['DaysToNearestHoliday'] = np.minimum(dist_prev, dist_next)

print(f"DaysToNearestHoliday: mean={dataset['DaysToNearestHoliday'].mean():.1f}, max={dataset['DaysToNearestHoliday'].max()}")

Total holiday dates generated: 12 across 1 years
Flights on holidays: 56,644 (3.0%)
DaysToNearestHoliday: mean=13.4, max=49


## Adding the Weather Data
With the Meteostat libary, 3 parameters for Weatehr data from the last full hour befor 2h beofer sheduled Departure.

In [ ]:
# load the airoorts dataset
airports = pd.read_csv('/content/drive/MyDrive/Datamining/Other_Data_files/airports_with_runway_info.csv')
# only keep the airports in the Airport List.
airports = airports[airports['iata_code'].isin(airport_limit_list)]

# merge the airports dataset with the flights dataset to get the timezone information for the departure and arrival airports
display(airports)
keep_cols = ['iata_code', 'type','scheduled_service','num_runways','most_common_surface','avg_runway_length','has_lighted_runways' ]
weather_cols = ['iata_code']
# get the colum index number of 'airport_station'
airport_station_index = airports.columns.get_loc('airport_station')
# add all columns from the index of 'airport_station' to the end of the dataframe to the list of columns to keep
weather_cols += airports.columns[airport_station_index:].tolist()



,type,name,latitude_deg,longitude_deg,elevation_ft,iso_country,iso_region,municipality,scheduled_service,icao_code,...,airport_data_length_code,closest_station_1,closest_station_1_distance,closest_station_1_name,closest_station_2,closest_station_2_distance,closest_station_2_name,closest_station_3,closest_station_3_distance,closest_station_3_name
5,large_airport,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101,1026.0,US,US-GA,Atlanta,True,KATL,...,1.0,KFTY0,18033.3,Atlanta / Carroll Heights,ULJ8M,28881.7,Henry County Airport,KPDK0,29009.9,Atlanta / Northwoods
7,large_airport,Austin Bergstrom International Airport,30.197535,-97.662015,542.0,US,US-TX,Austin,True,KAUS,...,1.0,74745,829.6,Austin / Del Valle,72254,16633.5,Camp Mabry/Austin City Asos,KEDC0,24055.5,Austin / New Sweden
15,large_airport,Nashville International Airport,36.124500,-86.678200,599.0,US,US-TN,Nashville,True,KBNA,...,1.0,72327,980.9,Nashville Airport,KMQY0,19154.1,Smyrna / Jefferson Pike,KJWN0,19796.0,John C Tune / Jordonia
17,large_airport,Boston Logan International Airport,42.361970,-71.007900,20.0,US,US-MA,Boston,True,KBOS,...,1.0,72509,894.1,Boston Logan International,74492,18466.8,Blue Hill Obs. Ma.,KOWD0,23405.6,Norwood
21,large_airport,Baltimore/Washington International Thurgood Ma...,39.175400,-76.668297,146.0,US,US-MD,Baltimore,True,KBWI,...,1.0,72406,1615.1,Baltimore-Washington International,KFME0,12724.0,Fort Meade / Ft Meade / Portland Station (Hist...,KDMH0,12794.6,Baltimore
27,large_airport,Charlotte Douglas International Airport,35.214001,-80.943100,748.0,US,US-NC,Charlotte,True,KCLT,...,1.0,72314,0.1,Charlotte/Douglas International Airport,KAKH0,18831.1,Gastonia / Forest Brook,KUZA0,27210.0,Rock Hill / Kimberly Woods
33,large_airport,Dallas Love Field,32.844776,-96.847653,487.0,US,US-TX,Dallas,True,KDAL,...,1.0,72258,465.7,Dallas / Oldham,KADS0,13808.6,Dallas / Addison,KNBE0,16658.0,Dallas / Lakeland Heights
35,large_airport,Ronald Reagan Washington National Airport,38.852100,-77.037697,15.0,US,US-DC,Washington,True,KDCA,...,1.0,72405,446.7,Washington National Airport,74594,16727.4,Camp Springs / Andrews Air Force Base,KCGS0,17431.0,College Park / Lakeland
36,large_airport,Denver International Airport,39.860027,-104.673792,5431.0,US,US-CO,Denver,True,KDEN,...,1.0,72565,957.6,Denver International Airport,KFTG0,13921.3,Colorado Air and Space Port,72469,18552.8,"Denver / Stapleton International, Co."
37,large_airport,Dallas Fort Worth International Airport,32.896801,-97.038002,607.0,US,US-TX,Dallas-Fort Worth,True,KDFW,...,1.0,72259,565.0,Dallas/Ft. Worth International Airport,72258,18246.6,Dallas / Oldham,KNBE0,19363.1,Dallas / Lakeland Heights


In [ ]:
ms.config.block_large_requests = False
airports_weather = airports[weather_cols]
# grop the flights dataset, and search for the fist and last flight date for each airport, and merge this information with the airports dataset
airport_flight_dates = dataset.groupby('Origin')['knownWeatherDateTime_UTC'].agg(['min', 'max']).reset_index()
# also do the same for the destination airports
airport_flight_dates_dest = dataset.groupby('Dest')['knownWeatherDateTime_UTC'].agg(['min', 'max']).reset_index()
# get the min out of both min dates, and the max out of both max dates, to get the date range for which we need weather data for each airport
airport_flight_dates = airport_flight_dates.merge(airport_flight_dates_dest, left_on='Origin', right_on='Dest', how='outer', suffixes=('_origin', '_dest'))

# subtract 5h from the min date, and add 5 hours to the max date, to get the date range for which we need weather data
airport_flight_dates['min'] = (airport_flight_dates[['min_origin', 'min_dest']].min(axis=1) - pd.Timedelta(hours=5)).dt.tz_localize(None) # get additional 5H and remove timezone info
airport_flight_dates['max'] = (airport_flight_dates[['max_origin', 'max_dest']].max(axis=1) + pd.Timedelta(hours=5)).dt.tz_localize(None) # get additional 5H and remove timezone info
airport_flight_dates = airport_flight_dates[['Origin', 'min', 'max']].rename(columns={'Origin': 'iata_code'})

# merge the airport flight dates with the airports weather dataset to get the date range for which we need weather data for each airport
airport_weather_dates = airport_flight_dates.merge(airports_weather, on='iata_code', how='inner')
# check if the length of all three datasets is the same, if not there are some airports for which we do not have weather data, and we need to drop them from the flights dataset
print(f"Length of airport_flight_dates: {len(airport_flight_dates)}")
print(f"Length of airports_weather: {len(airports_weather)}")
print(f"Length of airport_weather_dates: {len(airport_weather_dates)}")

Length of airport_flight_dates: 35
Length of airports_weather: 35
Length of airport_weather_dates: 35


In [ ]:
weather_params = ["temp","prcp","wspd"]
weather_data = pd.DataFrame()
print("Getting weather data for each airport, this may take a while...")

# function to fill the Weather Dataframe , with weatehr data.
def fill_weather_data(data, airport, date_range_length):
    data_length = len(data)
    # if the data is less than 95% complete, we try to fill it up with the data from the other near stations 1-3
    if data_length < date_range_length:
        print("filling up")
        # get data from another station
        for i in range(1,4):
            if airport[f'closest_station_{i}'] and airport[f'closest_station_{i}_distance'] < 50_000:
                data1 = ms.hourly(airport[f'closest_station_{i}'], airport['min'], airport['max'], parameters=weather_params)
                data1 = data1.fetch()
                if data1 is None or data1.empty:
                    continue
                data1["airport"] = airport["iata_code"]
                data1["timestamp"] = data1.index
                # match the data from the two stations and fill the missing rows in the first dataset with the values from the second dataset
                data = data.merge(data1, on=["timestamp", "airport"], how="outer", suffixes=("", "_1"))
                # fill every value on the first dataset with the value from the second dataset if it is missing in the first dataset
                for param in weather_params:
                    data[param] = data[param].fillna(data[f"{param}_1"])
                    data = data.drop(columns=[f"{param}_1"])
                if len(data) < date_range_length:
                    break
        print('finished filling up')
    return data

Getting weather data for each airport, this may take a while...


In [ ]:
# Collecting the Weather Data.
total_added_rows = 0
for index, airport in airport_weather_dates.iterrows():
    data_range_length = (airport['max'] - airport['min']).total_seconds() / 3600 # the number of Hours from max to min (how manny entries the Weather Data df will need)
    iata_code = airport['iata_code']

    data = None
    # base case: we have an aiport station, with weather data, which is mostly complete
    if airport['airport_station'] is not np.nan and airport['airport_data_length_code'] is not np.nan:
        #  get the Weather Data for every Hour
        data = ms.hourly(airport['airport_station'], airport['min'], airport['max'], parameters=weather_params)
        data = data.fetch()
        # if the Data is somehow empty, we create an empty df
        if data is None or data.empty:
            data = pd.DataFrame(columns=["timestamp", "airport"]+weather_params)
        # add columns to match the data on airport and hour later.
        data["airport"] = iata_code
        data["timestamp"] = data.index # the index is also the time, so this is essestially a copy

        # get the length of the data, and if it is less than 80% of the date range, we fill it up with the data from the other near stations 1-3
        data= fill_weather_data(data, airport, data_range_length) #calling the function
    else:
        data = pd.DataFrame(columns=["timestamp", "airport"]+weather_params)
        # if we do not have an airport station, we try to fill the data with the data from the other near stations 1-3
        data = fill_weather_data(data, airport, data_range_length)
    # now we have collected The Weather data



    # ---- This Block Makes shure for every Needed Hour there is a corresponding, row, even though it might be NULL for now
    # -> so we can performe a missing value operations later
    if len(data) <= data_range_length: # check if the data is incomplete, and there are Gaps
        #  we add rows, for the missing hours
        print(f" {iata_code} is incomplete. ({index+1}/{len(airport_weather_dates)}) --- {len(data)} rows ")
        # get the range of the data
        previ_length = len(data) #prev Or old Length.
        data_range = (airport['min'], airport['max'])
        # create a complete range of timestamps for the date range to make sure the entries for every hour exisit, even though might contain nothing
        complete_range = pd.date_range(start=data_range[0], end=data_range[1], freq='h')
        # make it a dataframe
        complete_range = pd.DataFrame(complete_range, columns=['timestamp'])
        complete_range['airport'] = airport['iata_code']
        # merge the complete range with the data to get the missing timestamps
        data = complete_range.merge(data, on=['timestamp', 'airport'], how='outer') # guarantee that for every hour, needed, there is an Hour.
        new_length = len(data)
        # print(f"added {new_length - previ_length} rows to the data for airport {iata_code}")
        total_added_rows += new_length - previ_length #track the added rows.
    else: # the Data is complete, and there are no holes.
        # print(f"{iata_code} is complete. ({index+1}/{len(airport_weather_dates)}) --- {len(data)} rows")
        pass



    # --- perform filling Nans. Standart method, is forward fill ( filling with the value of the last valid entry, with max 2 entries before)
    for param in weather_params:
        # try filling single nans with the previous value, if that is not also nan, we leave it as nan
        data[param] = data[param].ffill(limit=2)
    # hinterfragen wenn mit mehr daten gearbeite wird.
    # check if more than 50% of the values in the prcp column are missing, if so we fill them with 0, as it is more likely that there was no precipitation than that the data is missing


    # what to do if the forward fill is not enough, we will up differently.
    if data['prcp'].isna().sum() / len(data) > 0.5:
        data['prcp'] = data['prcp'].fillna(-1)
    else:
        data['prcp'] = data['prcp'].fillna(0)
    # same for the wspd
    if data['wspd'].isna().sum() / len(data) > 0.5:
        data['wspd'] = data['wspd'].fillna(-1)
    else:
        data['wspd'] = data['wspd'].fillna(0)
    data['temp'] = data['temp'].fillna(0)
    weather_data = pd.concat([weather_data, data], ignore_index=True)

 ATL is incomplete. (1/35) --- 8770 rows 
 AUS is incomplete. (2/35) --- 8763 rows 
filling up
finished filling up
 DAL is incomplete. (7/35) --- 8758 rows 
filling up
finished filling up
 DCA is incomplete. (8/35) --- 8763 rows 
 DEN is incomplete. (9/35) --- 8771 rows 
filling up
finished filling up
filling up
finished filling up
 FLL is incomplete. (13/35) --- 8765 rows 
filling up
finished filling up
 HOU is incomplete. (15/35) --- 8228 rows 
 IAH is incomplete. (16/35) --- 8769 rows 
 LAX is incomplete. (19/35) --- 8771 rows 
filling up
finished filling up
 LGA is incomplete. (20/35) --- 8760 rows 
 MDW is incomplete. (22/35) --- 8762 rows 
 MSP is incomplete. (24/35) --- 8766 rows 
filling up
finished filling up
 OAK is incomplete. (25/35) --- 8761 rows 
 ORD is incomplete. (26/35) --- 8767 rows 
filling up
finished filling up
 SAN is incomplete. (30/35) --- 8763 rows 
filling up
finished filling up
 SLC is incomplete. (33/35) --- 8765 rows 


In [ ]:
print(f"Total rows added: {total_added_rows}")
print(len(dataset))
display(weather_data)
#-  Merge the Weather Data onto the Departure and arrival Airport.
dataset = dataset.merge(weather_data, left_on=["Origin", "knownWeatherDateTime_UTC"], right_on=["airport", "timestamp"], how="left", suffixes=("", "_DEP"))
dataset = dataset.merge(weather_data, left_on=["Dest", "knownWeatherDateTime_UTC"], right_on=["airport", "timestamp"], how="left", suffixes=("", "_ARR"))

display(dataset)

Total rows added: 563
1867371


,timestamp,airport,temp,prcp,wspd
0,2014-01-01 00:00:00,ATL,4.4,0.0,7.6
1,2014-01-01 01:00:00,ATL,3.9,0.0,7.6
2,2014-01-01 02:00:00,ATL,3.3,0.0,7.6
3,2014-01-01 03:00:00,ATL,2.8,0.0,9.4
4,2014-01-01 04:00:00,ATL,2.8,0.0,9.4
...,...,...,...,...,...
306838,2015-01-01 05:00:00,TPA,15.6,0.0,11.2
306839,2015-01-01 06:00:00,TPA,15.0,0.0,13.0
306840,2015-01-01 07:00:00,TPA,15.0,0.0,13.0
306841,2015-01-01 08:00:00,TPA,15.6,0.0,11.2


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,timestamp,airport,temp,prcp,wspd,timestamp_ARR,airport_ARR,temp_ARR,prcp_ARR,wspd_ARR
0,2014,1,2,4,2014-01-02,DL,D942DN,785,MSP,ATL,...,2014-01-02 17:00:00,MSP,-19.4,0.0,5.4,2014-01-02 17:00:00,ATL,10.0,0.3,11.2
1,2014,1,3,5,2014-01-03,DL,D942DN,1939,ATL,AUS,...,2014-01-03 16:00:00,ATL,-3.9,0.0,18.4,2014-01-03 16:00:00,AUS,2.8,0.0,0.0
2,2014,1,3,5,2014-01-03,DL,D942DN,1642,AUS,ATL,...,2014-01-03 20:00:00,AUS,12.2,0.0,18.4,2014-01-03 20:00:00,ATL,1.1,0.0,11.2
3,2014,1,4,6,2014-01-04,DL,D942DN,2066,ATL,AUS,...,2014-01-04 15:00:00,ATL,-1.1,0.0,16.6,2014-01-04 15:00:00,AUS,8.9,0.0,13.0
4,2014,1,4,6,2014-01-04,DL,D942DN,2066,AUS,ATL,...,2014-01-04 19:00:00,AUS,19.4,0.0,33.5,2014-01-04 19:00:00,ATL,1.7,0.0,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867366,2014,12,28,7,2014-12-28,DL,N999DN,1782,ATL,LGA,...,2014-12-28 14:00:00,ATL,13.9,1.0,0.0,2014-12-28 14:00:00,LGA,7.2,1.3,14.8
1867367,2014,12,28,7,2014-12-28,DL,N999DN,221,LGA,ATL,...,2014-12-28 18:00:00,LGA,10.6,0.0,16.6,2014-12-28 18:00:00,ATL,15.6,0.0,9.4
1867368,2014,12,30,2,2014-12-30,DL,N999DN,2236,ATL,PHL,...,2014-12-30 16:00:00,ATL,10.0,0.0,14.8,2014-12-30 16:00:00,PHL,1.7,0.0,18.4
1867369,2014,12,30,2,2014-12-30,DL,N999DN,2236,PHL,ATL,...,2014-12-30 19:00:00,PHL,2.8,0.0,11.2,2014-12-30 19:00:00,ATL,10.0,0.0,22.3


In [ ]:
# get all the rows where the timestamp_Dep or timestamp_ARR is missing
missing_arr = dataset[dataset['timestamp_ARR'].isna()]
missing_dep = dataset[dataset['timestamp'].isna()]
# union of both missing datasets
missing = pd.concat([missing_arr, missing_dep])
# throw out duplicates
missing = missing.drop_duplicates()
display(missing)
# drop the rows where the timestamp_Dep is missing, as we can not use them for training
dataset = dataset[~dataset['timestamp_ARR'].isna()]
dataset = dataset[~dataset['timestamp'].isna()]

# drop the temporary collumns
dataset = dataset.drop(columns=['knownWeatherDateTime_UTC', 'timestamp', 'timestamp_ARR','airport','airport_ARR'])

# convert the temp, prcp and wspd columns to numeric, as they are currently object due to the nans
dataset['temp'] = pd.to_numeric(dataset['temp'], errors='coerce')
dataset['prcp'] = pd.to_numeric(dataset['prcp'], errors='coerce')
dataset['wspd'] = pd.to_numeric(dataset['wspd'], errors='coerce')
dataset['temp_ARR'] = pd.to_numeric(dataset['temp_ARR'], errors='coerce')
dataset['prcp_ARR'] = pd.to_numeric(dataset['prcp_ARR'], errors='coerce')
dataset['wspd_ARR'] = pd.to_numeric(dataset['wspd_ARR'], errors='coerce')

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,timestamp,airport,temp,prcp,wspd,timestamp_ARR,airport_ARR,temp_ARR,prcp_ARR,wspd_ARR


In [ ]:
# delete the temporary variables
del weather_data, airport_weather_dates, airports_weather, airports, weather_params

### Done with the Weather Data

## Additional Features
1. **Static time feature**: dep_hour
2. **Route column** for per-route aggregations
3. **Historical rolling delay features** (expanding + 7d + 30d means per airline/origin/dest/route) — 12 features
4. **Lag-1 / Lag-7 features** from autocorrelation analysis (per origin/dest/route/airline/global) — 10 features

In [ ]:
# === Static Time Feature ===
# dep_hour: captures peak-time delay patterns (raw CRSDepDateTime can't be fed to models)
dataset["dep_hour"] = dataset["CRSDepDateTime"].dt.hour

print(f"dep_hour: nunique={dataset['dep_hour'].nunique()}, null={dataset['dep_hour'].isna().sum()}")

dep_hour: nunique=22, null=0


In [ ]:
# === Historical Rolling Delay Features ===
# Expanding mean + 7-day + 30-day rolling mean of ArrDelayMinutes,
# grouped by airline, origin, and destination.
# shift(1) excludes the current day to prevent leakage.
# (Route grouping omitted — redundant with separate origin + dest features)

dataset = dataset.sort_values("CRSDepDateTime_UTC").reset_index(drop=True)

def compute_hist_features(df, daily_agg, group_col, prefix):
    """Compute expanding + 7d + 30d rolling means for a given grouping."""
    daily_agg = daily_agg.sort_values("FlightDate")
    col = f"{prefix}_daily_delay"

    daily_agg[f"hist_{prefix}_delay"] = (
        daily_agg.groupby(group_col)[col]
        .transform(lambda x: x.shift(1).expanding().mean()))
    daily_agg[f"hist_{prefix}_delay_7d"] = (
        daily_agg.groupby(group_col)[col]
        .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean()))
    daily_agg[f"hist_{prefix}_delay_30d"] = (
        daily_agg.groupby(group_col)[col]
        .transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean()))

    merge_cols = ["FlightDate", group_col,
                  f"hist_{prefix}_delay", f"hist_{prefix}_delay_7d", f"hist_{prefix}_delay_30d"]
    return df.merge(daily_agg[merge_cols], on=["FlightDate", group_col], how="left")

# Per airline
daily_airline = (dataset.groupby(["FlightDate", "Reporting_Airline"])["ArrDelayMinutes"]
                 .mean().reset_index()
                 .rename(columns={"ArrDelayMinutes": "airline_daily_delay"}))
dataset = compute_hist_features(dataset, daily_airline, "Reporting_Airline", "airline")

# Per origin airport
daily_origin = (dataset.groupby(["FlightDate", "Origin"])["ArrDelayMinutes"]
                .mean().reset_index()
                .rename(columns={"ArrDelayMinutes": "origin_daily_delay"}))
dataset = compute_hist_features(dataset, daily_origin, "Origin", "origin")

# Per destination airport
daily_dest = (dataset.groupby(["FlightDate", "Dest"])["ArrDelayMinutes"]
              .mean().reset_index()
              .rename(columns={"ArrDelayMinutes": "dest_daily_delay"}))
dataset = compute_hist_features(dataset, daily_dest, "Dest", "dest")

hist_features = [
    "hist_airline_delay", "hist_airline_delay_7d", "hist_airline_delay_30d",
    "hist_origin_delay", "hist_origin_delay_7d", "hist_origin_delay_30d",
    "hist_dest_delay", "hist_dest_delay_7d", "hist_dest_delay_30d",
]

# Fill NaN (first days have no history) with 0
for f in hist_features:
    dataset[f] = dataset[f].fillna(0)

print("Historical rolling features:")
for f in hist_features:
    print(f"  {f}: null={dataset[f].isna().sum():,}, mean={dataset[f].mean():.2f}")

del daily_airline, daily_origin, daily_dest

Historical rolling features:
  hist_airline_delay: null=0, mean=16.25
  hist_airline_delay_7d: null=0, mean=13.99
  hist_airline_delay_30d: null=0, mean=14.55
  hist_origin_delay: null=0, mean=16.46
  hist_origin_delay_7d: null=0, mean=14.08
  hist_origin_delay_30d: null=0, mean=14.65
  hist_dest_delay: null=0, mean=16.29
  hist_dest_delay_7d: null=0, mean=14.05
  hist_dest_delay_30d: null=0, mean=14.60


In [ ]:
# === Lag-1 / Lag-7 Features (from autocorrelation analysis) ===
# Yesterday's and last-week's mean delay as direct features.
# Lag-1 autocorrelation was ~0.53, lag-7 ~0.21 (weekly cycle).
# (Route grouping omitted — redundant with separate origin + dest features)

def add_lag_features(df, group_col, prefix):
    """Add lag-1 (yesterday) and lag-7 (last week) delay features for a grouping."""
    daily = (df.groupby(["FlightDate", group_col])["ArrDelayMinutes"]
             .mean().reset_index()
             .rename(columns={"ArrDelayMinutes": f"{prefix}_daily_mean"}))
    daily = daily.sort_values("FlightDate")

    daily[f"{prefix}_yesterday_delay"] = (
        daily.groupby(group_col)[f"{prefix}_daily_mean"].shift(1))
    daily[f"{prefix}_lastweek_delay"] = (
        daily.groupby(group_col)[f"{prefix}_daily_mean"].shift(7))

    merge_cols = ["FlightDate", group_col,
                  f"{prefix}_yesterday_delay", f"{prefix}_lastweek_delay"]
    return df.merge(daily[merge_cols], on=["FlightDate", group_col], how="left")

# Per origin airport
dataset = add_lag_features(dataset, "Origin", "origin")
# Per destination airport
dataset = add_lag_features(dataset, "Dest", "dest")
# Per airline
dataset = add_lag_features(dataset, "Reporting_Airline", "airline")

# Global lags (no grouping — overall system delay)
daily_global = (dataset.groupby("FlightDate")["ArrDelayMinutes"]
                .mean().reset_index()
                .rename(columns={"ArrDelayMinutes": "global_daily_mean"}))
daily_global = daily_global.sort_values("FlightDate")
daily_global["global_yesterday_delay"] = daily_global["global_daily_mean"].shift(1)
daily_global["global_lastweek_delay"] = daily_global["global_daily_mean"].shift(7)
dataset = dataset.merge(daily_global[["FlightDate", "global_yesterday_delay", "global_lastweek_delay"]],
                        on="FlightDate", how="left")

lag_features = [
    "origin_yesterday_delay", "origin_lastweek_delay",
    "dest_yesterday_delay", "dest_lastweek_delay",
    "airline_yesterday_delay", "airline_lastweek_delay",
    "global_yesterday_delay", "global_lastweek_delay",
]

# Fill NaN (first days have no prior data) with 0
for f in lag_features:
    dataset[f] = dataset[f].fillna(0)

print("Lag features:")
for f in lag_features:
    print(f"  {f}: null={dataset[f].isna().sum():,}, mean={dataset[f].mean():.2f}")

del daily_global

Lag features:
  origin_yesterday_delay: null=0, mean=13.84
  origin_lastweek_delay: null=0, mean=13.91
  dest_yesterday_delay: null=0, mean=13.82
  dest_lastweek_delay: null=0, mean=13.85
  airline_yesterday_delay: null=0, mean=13.76
  airline_lastweek_delay: null=0, mean=13.78
  global_yesterday_delay: null=0, mean=13.75
  global_lastweek_delay: null=0, mean=13.71


## Column Information

In [ ]:
# Collumns that are irrelavnt or contain leakage information, which can not be used in the model, that can be dropped:

# these contain information about the actual delay, not avaliabe at time of Prediction
target_like = ['DepDelay','DepDelayMinutes','ActualElapsedTime', 'Diverted', 'Cancelled','TaxiOut','TaxiIn','ArrTime','DepTime','ArrDateTime','DepDateTime','AirTime',
               'NASDelay','SecurityDelay','LateAircraftDelay','CarrierDelay','WeatherDelay','ArrDateTime_UTC','WheelsOff','WheelsOn','ArrDelay']
# Not relevant to training
timezone_cols = ['TZ_Origin', 'TZ_Dest']
# UTC Data, is irrelevant, and for Sheduled times, there are CRSDepDateTime columns still in the Dataframe.
timestamp_cols = ['DepDateTime_UTC', 'CRSArrDateTime_UTC', 'CRSDepTime', 'CRSArrTime','FlightDate']

# drop against overfitting
overtting_columns = ['FlightID','Tail_Number','Flight_Number_Reporting_Airline']

all_to_drop = target_like + timezone_cols + timestamp_cols + overtting_columns

In [ ]:
# create a second table, which just contains info about the features present in the dataset.
df_columns = pd.DataFrame({
    'data_type': dataset.dtypes,
    'unique_values': dataset.nunique(),
    'nan_values': dataset.isna().sum()
})

# 2. Add the column_name and Dropped status
# .index refers to the column names of 'dataset'
df_columns['column_name'] = df_columns.index
df_columns['Dropped'] = df_columns['column_name'].isin(all_to_drop)
# Nan Values
df_columns['nan_values'] = dataset.isna().sum()

df_columns["Category"] ="Undefined"
# traget_like collumns
df_columns.loc[df_columns['column_name']=="ArrDelayMinutes", 'Category'] = "Target"
df_columns.loc[df_columns['column_name'].isin(target_like), 'Category'] = "Target Like"


# for some Collumns we can write a Category to explain what they represent:
# 1. Flight Information, liek airports, airlines, tailnumbers and similar
flight_info_cols = ["Origin", "Dest", "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline","FlightID"]
df_columns.loc[df_columns['column_name'].isin(flight_info_cols), 'Category'] = "Flight Information"
# 2. Sheduled Time information in UTC and Local and the
time_info_cols = ["CRSDepDateTime","CRSArrDateTime","CRSDepDateTime_UTC", "CRSArrDateTime_UTC","knownWeatherDateTime_UTC",
                  'Year', 'Month','DayofMonth','DayOfWeek',"DayOfYear"]
df_columns.loc[df_columns['column_name'].isin(time_info_cols), 'Category'] = "Time Info"

# 3. Weather Data
weather_infor_cols = ["temp_ARR", "prcp_ARR", "wspd_ARR","temp_DEP", "prcp_DEP", "wspd_DEP"]
df_columns.loc[df_columns['column_name'].isin(weather_infor_cols), 'Category'] = "Weather_info"

# 4. real time statisitcal data collumns
real_time_cols = ["prev_AvgArrDelay","avg_delay"] # there are more, list is incomplete
df_columns.loc[df_columns['column_name'].isin(real_time_cols), 'Category'] = "Real time info"


# put the first and last value of the Collumn into another field, tohave examples



In [ ]:

# save the collumn info df
display(df_columns)
df_columns.to_csv(destination + "collumns.csv")


,data_type,unique_values,nan_values,column_name,Dropped,Category
Year,int64,1,0,Year,False,Time Info
Month,int64,12,0,Month,False,Time Info
DayofMonth,int64,31,0,DayofMonth,False,Time Info
DayOfWeek,int64,7,0,DayOfWeek,False,Time Info
FlightDate,datetime64[us],365,0,FlightDate,True,Undefined
...,...,...,...,...,...,...
dest_lastweek_delay,float64,11784,0,dest_lastweek_delay,False,Undefined
airline_yesterday_delay,float64,2546,0,airline_yesterday_delay,False,Undefined
airline_lastweek_delay,float64,2504,0,airline_lastweek_delay,False,Undefined
global_yesterday_delay,float64,365,0,global_yesterday_delay,False,Undefined


In [ ]:
# print all columns with nan values out
print("Columns with nan values:")
for col in dataset.columns:
  if dataset[col].isna().sum() > 0:
    print(f"colum: {col} has nan: {dataset[col].isna().sum()}")

Columns with nan values:
colum: CarrierDelay has nan: 1446660
colum: WeatherDelay has nan: 1446660
colum: NASDelay has nan: 1446660
colum: SecurityDelay has nan: 1446660
colum: LateAircraftDelay has nan: 1446660


In [ ]:
dataset.to_feather(destination + output_file_name)

In [ ]:
print("Saved the dataset to ")
print(destination + output_file_name)


Saved the dataset to 
/content/drive/MyDrive/Datamining/Feature_ingeneered_Data/feature_ingeneared.feather
